# pcb2bom: Pipeline Evaluation and Performance Analysis
This notebook provides an interactive walk-through and benchmark evaluation of the **pcb2bom** pipeline stages:
1. **Image Preprocessing:** Glare suppression & CLAHE color normalization
2. **Component Detection:** YOLO inference with OpenCV contour fallback
3. **OCR & Package Estimation:** IC laser etching extraction & geometric package heuristics
4. **Catalog Lookup:** RapidFuzz matching against mock database / DigiKey / Octopart
5. **BoM Export:** Aggregated engineering Bill of Materials with quantities and confidence metrics
6. **Precision & Recall Evaluation:** Real measured metrics on labeled ground truth

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from pcb2bom.preprocess import preprocess_image
from pcb2bom.detect import detect_components
from pcb2bom.ocr import run_ocr_pipeline
from pcb2bom.lookup import run_lookup_pipeline
from pcb2bom.bom import export_bom
from pcb2bom.pipeline import draw_component_annotations
from scripts.evaluate import evaluate_dataset

print("pcb2bom imports loaded successfully!")

## 1. Preprocessing Stage (Glare Inpainting & CLAHE)

In [ ]:
sample_img_path = PROJECT_ROOT / "data" / "sample_images" / "board_sample1.png"
prep_res = preprocess_image(sample_img_path, color_correction="clahe", fix_glare=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(prep_res.original, cv2.COLOR_BGR2RGB))
axes[0].set_title("Raw Input PCB Photo")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(prep_res.processed, cv2.COLOR_BGR2RGB))
axes[1].set_title("Preprocessed (CLAHE + Glare Suppression)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 2. Component Detection & Reference Designator Assignment

In [ ]:
components = detect_components(prep_res.processed)
print(f"Detected {len(components)} components:")
for c in components:
    print(f" - {c.id} ({c.class_name}): BoundingBox={c.bbox.as_tuple()}, Conf={c.confidence:.2f}")

## 3. Optical Character Recognition (OCR) & Package Heuristics

In [ ]:
components = run_ocr_pipeline(prep_res.processed, components)
for c in components:
    if c.ocr_result:
        print(f"{c.id}: OCR='{c.ocr_result.cleaned_text}', EstPackage='{c.ocr_result.package_estimate}'")

## 4. Part Lookup & Fuzzy Catalog Matching

In [ ]:
mock_db = PROJECT_ROOT / "data" / "sample_parts_mock.json"
components, unidentified = run_lookup_pipeline(components, use_mock=True, mock_file_path=mock_db)

for c in components:
    if c.lookup_result:
        print(f"{c.id} -> MPN: {c.lookup_result.matched_mpn} | Mfg: {c.lookup_result.manufacturer} | MatchConf: {c.lookup_result.match_confidence:.2f}")

## 5. Aggregated Engineering BoM Generation

In [ ]:
out_dir = PROJECT_ROOT / "output"
csv_p, xlsx_p, df_bom = export_bom(components, output_dir=out_dir)
df_bom

## 6. Real Measured Benchmark Evaluation

In [ ]:
ann_file = PROJECT_ROOT / "data" / "annotations" / "ground_truth.json"
img_folder = PROJECT_ROOT / "data" / "sample_images"

metrics = evaluate_dataset(ann_file, img_folder, iou_threshold=0.40)

print("=== REAL MEASURED EVALUATION REPORT ===")
print(f"Ground Truth Components : {metrics.total_ground_truth_components}")
print(f"Detected Components     : {metrics.total_detected_components}")
print(f"True Positives (TP)     : {metrics.true_positives}")
print(f"False Positives (FP)    : {metrics.false_positives}")
print(f"False Negatives (FN)    : {metrics.false_negatives}")
print(f"Precision               : {metrics.precision:.1%}")
print(f"Recall                  : {metrics.recall:.1%}")
print(f"F1 Score                : {metrics.f1_score:.1%}")
print(f"ICs Correctly OCR'd     : {metrics.ics_correctly_ocred}/{metrics.total_ics_ground_truth} ({metrics.ocr_accuracy:.1%})")
print(f"ICs Identified          : {metrics.ics_correctly_identified}/{metrics.total_ics_ground_truth} ({metrics.identification_accuracy:.1%})")